## 1. Setup

In [1]:
import glob
import numpy as np
import pandas as pd

SEED = 103
DATA_DIR = "/kaggle/input"

pd.set_option("display.max_columns", 50)

for path in sorted(glob.glob(f"{DATA_DIR}/**/*.csv", recursive=True)):
    print(path)

/kaggle/input/competitions/predictive-modelling-ds/creators_daily.csv
/kaggle/input/competitions/predictive-modelling-ds/engagement_daily.csv
/kaggle/input/competitions/predictive-modelling-ds/sample_submission.csv
/kaggle/input/competitions/predictive-modelling-ds/test_videos.csv
/kaggle/input/competitions/predictive-modelling-ds/train_videos.csv


## 2. Load Data

In [2]:
def load_csv_table(filename):
    matches = glob.glob(f"{DATA_DIR}/**/{filename}", recursive=True)
    if not matches:
        raise FileNotFoundError(filename)
    return pd.read_csv(matches[0])


train_videos = load_csv_table("train_videos.csv")
test_videos = load_csv_table("test_videos.csv")
engagement_daily = load_csv_table("engagement_daily.csv")
creators_daily = load_csv_table("creators_daily.csv")
sample_submission = load_csv_table("sample_submission.csv")

raw_tables = {
    "train_videos": train_videos,
    "test_videos": test_videos,
    "engagement_daily": engagement_daily,
    "creators_daily": creators_daily,
    "sample_submission": sample_submission,
}

for table_name, table in raw_tables.items():
    print(table_name, table.shape)
    print(table.columns.tolist())

train_videos (12000, 27)
['video_id', 'author_id', 'create_time', 'create_date', 'duration', 'ratio', 'desc_language', 'is_english', 'created_by_ai', 'is_ads', 'music_selected_from', 'music_author', 'music_owner_id', 'music_id', 'word_count', 'emoji_count', 'question_count', 'hashtag_count', 'speaking_rate', 'topic', 'anger', 'joy', 'surprise', 'sadness', 'disgust', 'fear', 'target_day30_views']
test_videos (3001, 26)
['video_id', 'author_id', 'create_time', 'create_date', 'duration', 'ratio', 'desc_language', 'is_english', 'created_by_ai', 'is_ads', 'music_selected_from', 'music_author', 'music_owner_id', 'music_id', 'word_count', 'emoji_count', 'question_count', 'hashtag_count', 'speaking_rate', 'topic', 'anger', 'joy', 'surprise', 'sadness', 'disgust', 'fear']
engagement_daily (79489, 10)
['video_id', 'date', 'days_since_post', 'play_count', 'like_count', 'comment_count', 'share_count', 'collect_count', 'download_count', 'whatsapp_share_count']
creators_daily (252166, 7)
['author_id

## 3. Quick Data Check

In [3]:
day30_views = train_videos["target_day30_views"]

print(day30_views.describe())
print(day30_views.quantile([0.5, 0.75, 0.9, 0.99, 0.999]))
print("videos with zero views:", (day30_views == 0).sum())

count    1.200000e+04
mean     2.922860e+04
std      2.752131e+05
min      0.000000e+00
25%      3.440000e+02
50%      6.680000e+02
75%      3.016500e+03
max      1.631055e+07
Name: target_day30_views, dtype: float64
0.500        668.000
0.750       3016.500
0.900      21729.800
0.990     544054.870
0.999    4398992.766
Name: target_day30_views, dtype: float64
videos with zero views: 2


In [4]:
days_per_video = engagement_daily.groupby("video_id").size()

print("videos in engagement_daily:", days_per_video.shape[0])
print(days_per_video.value_counts().sort_index())
print(engagement_daily["days_since_post"].value_counts().sort_index())

all_video_ids = pd.concat([train_videos["video_id"], test_videos["video_id"]])
print("videos with no engagement rows:", (~all_video_ids.isin(engagement_daily["video_id"])).sum())

videos in engagement_daily: 15000
1       1
2       2
3      14
4     643
5    9170
6    5170
Name: count, dtype: int64
days_since_post
0     5638
1    14750
2    14726
3    14768
4    14856
5    14751
Name: count, dtype: int64
videos with no engagement rows: 1


## 4. Engagement by Day (One Row per Video)

In [5]:
ENGAGEMENT_METRICS = [
    "play_count",
    "like_count",
    "comment_count",
    "share_count",
    "collect_count",
    "download_count",
    "whatsapp_share_count",
]

duplicate_rows = engagement_daily.duplicated(["video_id", "days_since_post"]).sum()
print("duplicate (video, day) rows:", duplicate_rows)

engagement_daily_wide = engagement_daily.pivot(
    index="video_id",
    columns="days_since_post",
    values=ENGAGEMENT_METRICS,
)

engagement_daily_wide.columns = [
    f"{metric}_day{day}" for metric, day in engagement_daily_wide.columns
]
engagement_daily_wide = engagement_daily_wide.reset_index()

print(engagement_daily_wide.shape)
print(engagement_daily_wide.head())

duplicate (video, day) rows: 0
(15000, 43)
              video_id  play_count_day0  play_count_day1  play_count_day2  \
0  7384086821132717358         105696.0         330084.0         390753.0   
1  7384175589206265118            138.0            323.0            388.0   
2  7384177529092951338          10524.0          11045.0          11290.0   
3  7384193780057984287            273.0            356.0            378.0   
4  7384193851700956447            541.0           1414.0           1959.0   

   play_count_day3  play_count_day4  play_count_day5  like_count_day0  \
0         464862.0         475386.0         478085.0           6249.0   
1            447.0            492.0            557.0              7.0   
2          11430.0          11610.0          11743.0           1388.0   
3            397.0            410.0            423.0             36.0   
4           2258.0           2467.0           2697.0             16.0   

   like_count_day1  like_count_day2  like_count_day3  l

## 5. Attach Engagement to Videos

In [6]:
train_videos_with_engagement = train_videos.merge(
    engagement_daily_wide,
    on="video_id",
    how="left",
    validate="one_to_one",
)
test_videos_with_engagement = test_videos.merge(
    engagement_daily_wide,
    on="video_id",
    how="left",
    validate="one_to_one",
)

print(train_videos.shape, "->", train_videos_with_engagement.shape)
print(test_videos.shape, "->", test_videos_with_engagement.shape)

print(train_videos_with_engagement.filter(like="play_count").isna().sum())

(12000, 27) -> (12000, 69)
(3001, 26) -> (3001, 68)
play_count_day0    7539
play_count_day1     187
play_count_day2     227
play_count_day3     182
play_count_day4     112
play_count_day5     210
dtype: int64


## 6. Engagement Rates

In [7]:
RATE_DAY = 5

ENGAGEMENT_RATES = {
    "likes_per_view": "like_count",
    "comments_per_view": "comment_count",
    "shares_per_view": "share_count",
    "collects_per_view": "collect_count",
}


def add_engagement_rates(table, day):
    table = table.copy()
    views = table[f"play_count_day{day}"]
    views = views.where(views > 0)
    for rate_name, count_column in ENGAGEMENT_RATES.items():
        table[f"{rate_name}_day{day}"] = table[f"{count_column}_day{day}"] / views
    return table


train_videos_with_engagement = add_engagement_rates(train_videos_with_engagement, RATE_DAY)
test_videos_with_engagement = add_engagement_rates(test_videos_with_engagement, RATE_DAY)

print(train_videos_with_engagement.shape)
print(test_videos_with_engagement.shape)
print(train_videos_with_engagement.filter(like="per_view").describe())

(12000, 73)
(3001, 72)
       likes_per_view_day5  comments_per_view_day5  shares_per_view_day5  \
count         11786.000000            11786.000000          11786.000000   
mean              0.079257                0.007644              0.002506   
std               0.057765                0.016167              0.008598   
min               0.000000                0.000000              0.000000   
25%               0.036588                0.000000              0.000000   
50%               0.069444                0.002621              0.000000   
75%               0.110123                0.007961              0.002188   
max               2.000000                0.345455              0.296849   

       collects_per_view_day5  
count            11786.000000  
mean                 0.004512  
std                  0.007437  
min                  0.000000  
25%                  0.000000  
50%                  0.002487  
75%                  0.005602  
max                  0.250000  


## 7. Creator Data Check

In [8]:
print(creators_daily["date"].min(), creators_daily["date"].max())
print(creators_daily.groupby("author_id").size().describe())

print("train create_date:", train_videos["create_date"].min(), train_videos["create_date"].max())
print("test create_date:", test_videos["create_date"].min(), test_videos["create_date"].max())

print("test videos whose creator is in train:", test_videos["author_id"].isin(train_videos["author_id"]).mean())

for table_name, table in {"train_videos": train_videos, "test_videos": test_videos}.items():
    print(table_name, "creator found in creators_daily:", table["author_id"].isin(creators_daily["author_id"]).mean())

print(creators_daily.isna().sum())

2024-06-24 2024-12-09
count    1671.000000
mean      150.907241
std        17.196167
min        48.000000
25%       151.000000
50%       158.000000
75%       160.000000
max       161.000000
dtype: float64
train create_date: 2024-06-24 2024-11-09
test create_date: 2024-06-24 2024-11-09
test videos whose creator is in train: 0.9746751082972342
train_videos creator found in creators_daily: 1.0
test_videos creator found in creators_daily: 1.0
author_id                   0
date                        0
follower_count            166
following_count           166
total_favorited             0
video_count                25
enterprise_verified    252166
dtype: int64


## 8. Uncomfortable Content Score

In [9]:
UNCOMFORTABLE_EMOTIONS = ["fear", "disgust", "anger", "sadness"]


def add_uncomfortable_content_score(table, emotion_columns):
    table = table.copy()
    table["uncomfortable_content_score"] = table[emotion_columns].sum(axis=1, skipna=False)
    return table


train_videos_with_engagement = add_uncomfortable_content_score(train_videos_with_engagement, UNCOMFORTABLE_EMOTIONS)
test_videos_with_engagement = add_uncomfortable_content_score(test_videos_with_engagement, UNCOMFORTABLE_EMOTIONS)

print(train_videos_with_engagement[UNCOMFORTABLE_EMOTIONS + ["uncomfortable_content_score"]].describe())
print(train_videos_with_engagement[["uncomfortable_content_score", "joy", "target_day30_views"]].corr(method="spearman"))

               fear       disgust         anger       sadness  \
count  11958.000000  11958.000000  11958.000000  11958.000000   
mean       0.019192      0.053046      0.026587      0.029333   
std        0.099781      0.110757      0.106187      0.117090   
min        0.000180      0.000209      0.000514      0.000685   
25%        0.000596      0.008631      0.003416      0.003189   
50%        0.001101      0.018254      0.005442      0.004832   
75%        0.002205      0.044833      0.008791      0.009292   
max        0.988201      0.990008      0.986903      0.989842   

       uncomfortable_content_score  
count                 11958.000000  
mean                      0.128157  
std                       0.225326  
min                       0.003297  
25%                       0.018872  
50%                       0.036524  
75%                       0.095318  
max                       0.995836  
                             uncomfortable_content_score       joy  \
uncomfortab

## 9. Creator Features

In [10]:
CREATOR_COLUMNS = [
    "follower_count",
    "following_count",
    "total_favorited",
    "video_count",
]
CREATOR_START_DAY = 0
CREATOR_END_DAY = 5

assert CREATOR_END_DAY <= 5

creators_daily["date"] = pd.to_datetime(creators_daily["date"])

print("duplicate (creator, date) rows:", creators_daily.duplicated(["author_id", "date"]).sum())


def add_creator_features(videos, creators):
    videos = videos.drop(columns=videos.filter(like="creator_").columns)
    post_date = pd.to_datetime(videos["create_date"])
    for offset in (CREATOR_START_DAY, CREATOR_END_DAY):
        videos["creator_lookup_date"] = post_date + pd.Timedelta(days=offset)
        rename_map = {column: f"creator_{column}_day{offset}" for column in CREATOR_COLUMNS}
        rename_map["date"] = "creator_lookup_date"
        creator_lookup = creators[["author_id", "date"] + CREATOR_COLUMNS].rename(columns=rename_map)
        videos = videos.merge(
            creator_lookup,
            on=["author_id", "creator_lookup_date"],
            how="left",
            validate="many_to_one",
        )
    videos = videos.drop(columns="creator_lookup_date")
    for column in CREATOR_COLUMNS:
        end_value = videos[f"creator_{column}_day{CREATOR_END_DAY}"]
        start_value = videos[f"creator_{column}_day{CREATOR_START_DAY}"]
        videos[f"creator_{column}_gain"] = end_value - start_value
    return videos


def add_views_per_follower(videos, day):
    videos = videos.copy()
    followers = videos[f"creator_follower_count_day{CREATOR_START_DAY}"]
    followers = followers.where(followers > 0)
    videos[f"views_per_follower_day{day}"] = videos[f"play_count_day{day}"] / followers
    return videos


train_videos_with_engagement = add_creator_features(train_videos_with_engagement, creators_daily)
test_videos_with_engagement = add_creator_features(test_videos_with_engagement, creators_daily)

train_videos_with_engagement = add_views_per_follower(train_videos_with_engagement, RATE_DAY)
test_videos_with_engagement = add_views_per_follower(test_videos_with_engagement, RATE_DAY)

print(train_videos_with_engagement.shape)
print(test_videos_with_engagement.shape)
print(train_videos_with_engagement.filter(like="creator_").isna().sum())

CHECK_COLUMNS = [
    "creator_follower_count_day0",
    "views_per_follower_day5",
    "comments_per_view_day5",
    "likes_per_view_day5",
    "play_count_day5",
    "target_day30_views",
]
print(train_videos_with_engagement[CHECK_COLUMNS].corr(method="spearman")["target_day30_views"])

duplicate (creator, date) rows: 0
(12000, 87)
(3001, 86)
creator_follower_count_day0      0
creator_following_count_day0     0
creator_total_favorited_day0     0
creator_video_count_day0         0
creator_follower_count_day5     15
creator_following_count_day5    15
creator_total_favorited_day5     7
creator_video_count_day5         7
creator_follower_count_gain     15
creator_following_count_gain    15
creator_total_favorited_gain     7
creator_video_count_gain         7
dtype: int64
creator_follower_count_day0    0.642228
views_per_follower_day5        0.279709
comments_per_view_day5        -0.014248
likes_per_view_day5           -0.008698
play_count_day5                0.985903
target_day30_views             1.000000
Name: target_day30_views, dtype: float64


## 10. Growth After Day 5 Check

In [11]:
views_day5 = train_videos_with_engagement["play_count_day5"]
views_day5 = views_day5.where(views_day5 > 0)
views_day4 = train_videos_with_engagement["play_count_day4"]
target_views = train_videos_with_engagement["target_day30_views"]

growth_factor = target_views / views_day5
last_day_views_share = (views_day5 - views_day4) / views_day5

print(growth_factor.describe())
print(growth_factor.quantile([0.01, 0.1, 0.5, 0.9, 0.99]))
print("videos where day 30 views are below day 5 views:", (growth_factor < 1).sum())

CANDIDATE_FEATURES = [
    "comments_per_view_day5",
    "likes_per_view_day5",
    "shares_per_view_day5",
    "collects_per_view_day5",
    "uncomfortable_content_score",
    "joy",
    "creator_follower_count_day0",
    "creator_follower_count_gain",
    "views_per_follower_day5",
    "play_count_day5",
]

growth_check_table = train_videos_with_engagement[CANDIDATE_FEATURES].copy()
growth_check_table["last_day_views_share"] = last_day_views_share
growth_check_table["growth_factor"] = growth_factor

print(growth_check_table.corr(method="spearman")["growth_factor"].drop("growth_factor").sort_values())

count    11786.000000
mean         1.627504
std         10.257513
min          0.000000
25%          1.080769
50%          1.161795
75%          1.335232
max        767.269551
dtype: float64
0.01    1.008482
0.10    1.039832
0.50    1.161795
0.90    1.687500
0.99    4.772162
dtype: float64
videos where day 30 views are below day 5 views: 1
views_per_follower_day5       -0.132117
uncomfortable_content_score   -0.073885
comments_per_view_day5         0.040321
joy                            0.047408
likes_per_view_day5            0.106480
collects_per_view_day5         0.147961
shares_per_view_day5           0.182186
creator_follower_count_gain    0.220809
play_count_day5                0.335808
creator_follower_count_day0    0.374751
last_day_views_share           0.746519
Name: growth_factor, dtype: float64


## 11. Last Known Views and Growth Factor

In [12]:
PLAY_COUNT_COLUMNS = [f"play_count_day{day}" for day in range(6)]
DAY_NUMBERS = np.arange(6)


def add_last_known_views(videos):
    videos = videos.copy()
    play_counts = videos[PLAY_COUNT_COLUMNS]
    has_any_views = play_counts.notna().any(axis=1)
    last_known_day = (play_counts.notna().to_numpy() * DAY_NUMBERS).max(axis=1)
    videos["last_known_views"] = play_counts.ffill(axis=1).iloc[:, -1]
    videos["last_known_day"] = pd.Series(last_known_day, index=videos.index).where(has_any_views)
    return videos


def add_growth_factor(videos):
    videos = videos.copy()
    known_views = videos["last_known_views"].where(videos["last_known_views"] > 0)
    videos["growth_factor"] = videos["target_day30_views"] / known_views
    return videos


train_videos_with_engagement = add_last_known_views(train_videos_with_engagement)
test_videos_with_engagement = add_last_known_views(test_videos_with_engagement)
train_videos_with_engagement = add_growth_factor(train_videos_with_engagement)

print(train_videos_with_engagement["last_known_day"].value_counts(dropna=False).sort_index())
print(test_videos_with_engagement["last_known_day"].value_counts(dropna=False).sort_index())

print("train videos with no usable last known views:", train_videos_with_engagement["growth_factor"].isna().sum())
print("test videos with no usable last known views:", (~(test_videos_with_engagement["last_known_views"] > 0)).sum())

day5_rows = train_videos_with_engagement["last_known_day"] == 5
same_as_day5 = train_videos_with_engagement.loc[day5_rows, "last_known_views"] == train_videos_with_engagement.loc[day5_rows, "play_count_day5"]
print("day 5 rows where last_known_views equals play_count_day5:", same_as_day5.all())

print(train_videos_with_engagement.groupby("last_known_day")["growth_factor"].median())

last_known_day
3.0        4
4.0      205
5.0    11790
NaN        1
Name: count, dtype: int64
last_known_day
0       1
4      39
5    2961
Name: count, dtype: int64
train videos with no usable last known views: 6
test videos with no usable last known views: 0
day 5 rows where last_known_views equals play_count_day5: True
last_known_day
3.0    1.061964
4.0    1.173725
5.0    1.161795
Name: growth_factor, dtype: float64


## 12. Momentum Features

In [13]:
def add_momentum_features(videos):
    videos = videos.copy()
    views_day1 = videos["play_count_day1"].where(videos["play_count_day1"] > 0)
    views_day3 = videos["play_count_day3"]
    views_day4 = videos["play_count_day4"]
    views_day5 = videos["play_count_day5"].where(videos["play_count_day5"] > 0)
    videos["last_day_views_share"] = (views_day5 - views_day4) / views_day5
    videos["last_two_days_views_share"] = (views_day5 - views_day3) / views_day5
    videos["views_growth_day1_to_day5"] = views_day5 / views_day1
    return videos


train_videos_with_engagement = add_momentum_features(train_videos_with_engagement)
test_videos_with_engagement = add_momentum_features(test_videos_with_engagement)

MOMENTUM_FEATURES = [
    "last_day_views_share",
    "last_two_days_views_share",
    "views_growth_day1_to_day5",
]

print(train_videos_with_engagement.shape)
print(test_videos_with_engagement.shape)
print(train_videos_with_engagement[MOMENTUM_FEATURES].describe())
print(train_videos_with_engagement[MOMENTUM_FEATURES + ["growth_factor"]].corr(method="spearman")["growth_factor"])

(12000, 93)
(3001, 91)
       last_day_views_share  last_two_days_views_share  \
count          11679.000000               11605.000000   
mean               0.033725                   0.076241   
std                0.046626                   0.087647   
min                0.000000                   0.000000   
25%                0.008863                   0.023191   
50%                0.020528                   0.049180   
75%                0.042147                   0.098519   
max                1.000000                   1.000000   

       views_growth_day1_to_day5  
count               11598.000000  
mean                    1.552036  
std                     2.525691  
min                     1.000000  
25%                     1.103541  
50%                     1.235046  
75%                     1.530866  
max                   144.925258  
last_day_views_share         0.746519
last_two_days_views_share    0.768088
views_growth_day1_to_day5    0.701988
growth_factor            

## 13. Video Columns Check

In [14]:
VIDEO_COLUMNS = [
    "create_time",
    "duration",
    "ratio",
    "desc_language",
    "is_english",
    "created_by_ai",
    "is_ads",
    "music_selected_from",
    "music_author",
    "music_owner_id",
    "music_id",
    "word_count",
    "emoji_count",
    "question_count",
    "hashtag_count",
    "speaking_rate",
    "topic",
]

column_summary = pd.DataFrame({
    "dtype": train_videos[VIDEO_COLUMNS].dtypes.astype(str),
    "missing": train_videos[VIDEO_COLUMNS].isna().sum(),
    "unique_values": train_videos[VIDEO_COLUMNS].nunique(),
})
print(column_summary)

for column in ["ratio", "desc_language", "is_english", "created_by_ai", "is_ads", "music_selected_from", "topic"]:
    print(train_videos[column].value_counts(dropna=False).head(10))
    print()

print(train_videos["create_time"].head())
print(train_videos["duration"].describe())

for column in ["music_author", "music_owner_id", "music_id", "topic"]:
    print(column, "share of test videos whose value appears in train:", test_videos[column].isin(train_videos[column]).mean())

                       dtype  missing  unique_values
create_time           object        0          11975
duration             float64        0           5980
ratio                 object        0              5
desc_language         object        0              2
is_english             int64        0              2
created_by_ai        float64        0              2
is_ads                 int64        0              1
music_selected_from   object      121             39
music_author          object       21           6187
music_owner_id       float64     2972           4518
music_id             float64        1          10376
word_count           float64     4047            489
emoji_count          float64     4047             25
question_count       float64     4047             22
hashtag_count        float64     4047             51
speaking_rate        float64     4047           3386
topic                 object        0              9
ratio
540p     11131
720p       821
480p      

## 14. Video Features and Final Feature List

In [15]:
CATEGORICAL_COLUMNS = ["topic", "music_selected_from"]
VIDEO_NEW_COLUMNS = ["post_hour", "post_weekday", "resolution_height"]


def add_video_features(videos):
    videos = videos.copy()
    post_time = pd.to_datetime(videos["create_time"])
    videos["post_hour"] = post_time.dt.hour
    videos["post_weekday"] = post_time.dt.dayofweek
    resolution_text = videos["ratio"].str.replace("p", "", regex=False)
    videos["resolution_height"] = pd.to_numeric(resolution_text, errors="coerce")
    return videos


def set_category_types(train_table, test_table, columns):
    train_table = train_table.copy()
    test_table = test_table.copy()
    for column in columns:
        categories = sorted(train_table[column].dropna().unique())
        category_type = pd.CategoricalDtype(categories=categories)
        train_table[column] = train_table[column].astype(category_type)
        test_table[column] = test_table[column].astype(category_type)
    return train_table, test_table


train_videos_with_engagement = add_video_features(train_videos_with_engagement)
test_videos_with_engagement = add_video_features(test_videos_with_engagement)

train_videos_with_engagement, test_videos_with_engagement = set_category_types(
    train_videos_with_engagement,
    test_videos_with_engagement,
    CATEGORICAL_COLUMNS,
)

FEATURE_GROUPS = {
    "base": ["last_known_views", "last_known_day"],
    "daily_counts": engagement_daily_wide.columns.drop("video_id").tolist(),
    "momentum": MOMENTUM_FEATURES,
    "engagement_rates": [f"{rate_name}_day{RATE_DAY}" for rate_name in ENGAGEMENT_RATES],
    "creator": (
        [f"creator_{column}_day{CREATOR_START_DAY}" for column in CREATOR_COLUMNS]
        + [f"creator_{column}_day{CREATOR_END_DAY}" for column in CREATOR_COLUMNS]
        + [f"creator_{column}_gain" for column in CREATOR_COLUMNS]
        + [f"views_per_follower_day{RATE_DAY}"]
    ),
    "emotions": ["anger", "joy", "surprise", "sadness", "disgust", "fear", "uncomfortable_content_score"],
    "video": [
        "duration", "is_english", "created_by_ai",
        "word_count", "emoji_count", "question_count", "hashtag_count", "speaking_rate",
        "post_hour", "post_weekday", "resolution_height",
        "topic", "music_selected_from",
    ],
}

ALL_FEATURES = []
for feature_names in FEATURE_GROUPS.values():
    ALL_FEATURES.extend(feature_names)

FORBIDDEN_FEATURES = ["target_day30_views", "growth_factor", "video_id", "author_id"]

assert len(ALL_FEATURES) == len(set(ALL_FEATURES))
assert not set(ALL_FEATURES) & set(FORBIDDEN_FEATURES)
assert set(ALL_FEATURES) <= set(train_videos_with_engagement.columns)
assert set(ALL_FEATURES) <= set(test_videos_with_engagement.columns)

feature_days = [int(name.split("_day")[-1]) for name in ALL_FEATURES if name.split("_day")[-1].isdigit()]
assert max(feature_days) <= 5

print(train_videos_with_engagement.shape)
print(test_videos_with_engagement.shape)
print("total features:", len(ALL_FEATURES))
for group_name, feature_names in FEATURE_GROUPS.items():
    print(group_name, len(feature_names))

print(train_videos_with_engagement[VIDEO_NEW_COLUMNS].describe())
print(train_videos_with_engagement[CATEGORICAL_COLUMNS].dtypes)
print(train_videos_with_engagement[CATEGORICAL_COLUMNS].isna().sum())
print(test_videos_with_engagement[CATEGORICAL_COLUMNS].isna().sum())

(12000, 96)
(3001, 94)
total features: 84
base 2
daily_counts 42
momentum 3
engagement_rates 4
creator 13
emotions 7
video 13
          post_hour  post_weekday  resolution_height
count  12000.000000   12000.00000       12000.000000
mean      19.564000       2.83450         551.995000
std        2.037421       1.96607          46.288212
min        1.000000       0.00000         360.000000
25%       18.000000       1.00000         540.000000
50%       19.000000       3.00000         540.000000
75%       21.000000       4.00000         540.000000
max       23.000000       6.00000        1080.000000
topic                  category
music_selected_from    category
dtype: object
topic                    0
music_selected_from    121
dtype: int64
topic                   0
music_selected_from    32
dtype: int64


## 15. Local Validation and Simple Baselines

In [16]:
from sklearn.model_selection import KFold

N_FOLDS = 5

usable_train_videos = train_videos_with_engagement[
    train_videos_with_engagement["growth_factor"].notna()
].reset_index(drop=True)

fold_splitter = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)


def compute_rmse(actual, predicted):
    difference = np.asarray(actual, dtype=float) - np.asarray(predicted, dtype=float)
    return float(np.sqrt(np.mean(difference ** 2)))


def compute_rmse_on_log_scale(actual, predicted):
    return compute_rmse(np.log1p(actual), np.log1p(predicted))


def evaluate_simple_baselines(videos, splitter):
    fold_results = []
    for fold_number, (fit_index, holdout_index) in enumerate(splitter.split(videos), start=1):
        fit_part = videos.iloc[fit_index]
        holdout_part = videos.iloc[holdout_index]
        median_growth = fit_part["growth_factor"].median()
        actual_views = holdout_part["target_day30_views"]
        same_as_last_known = holdout_part["last_known_views"]
        last_known_times_median = holdout_part["last_known_views"] * median_growth
        fold_results.append({
            "fold": fold_number,
            "median_growth": median_growth,
            "rmse_same_as_last_known": compute_rmse(actual_views, same_as_last_known),
            "rmse_last_known_times_median": compute_rmse(actual_views, last_known_times_median),
            "log_rmse_same_as_last_known": compute_rmse_on_log_scale(actual_views, same_as_last_known),
            "log_rmse_last_known_times_median": compute_rmse_on_log_scale(actual_views, last_known_times_median),
        })
    return pd.DataFrame(fold_results)


baseline_results = evaluate_simple_baselines(usable_train_videos, fold_splitter)

print("usable train videos:", len(usable_train_videos))
print(baseline_results.round(3))
print(baseline_results.drop(columns="fold").agg(["mean", "std"]).round(3))
print("target standard deviation:", round(usable_train_videos["target_day30_views"].std(), 1))

usable train videos: 11994
   fold  median_growth  rmse_same_as_last_known  rmse_last_known_times_median  \
0     1          1.162               103400.996                     87322.633   
1     2          1.161                95351.099                     76439.374   
2     3          1.163                88469.678                     75483.701   
3     4          1.162               111543.522                     77673.123   
4     5          1.162                70725.448                     55233.170   

   log_rmse_same_as_last_known  log_rmse_last_known_times_median  
0                        0.387                             0.318  
1                        0.409                             0.341  
2                        0.473                             0.414  
3                        0.435                             0.368  
4                        0.409                             0.341  
      median_growth  rmse_same_as_last_known  rmse_last_known_times_median  \
mean  

## 16. LightGBM Growth Model (All Features)

In [17]:
import lightgbm as lgb

LIGHTGBM_PARAMS = {
    "n_estimators": 300,
    "learning_rate": 0.03,
    "num_leaves": 15,
    "min_child_samples": 30,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.7,
    "random_state": SEED,
    "verbose": -1,
}

MINIMUM_GROWTH = 1.0
TARGET_NAMES = ["growth", "log_growth"]
WEIGHT_SCHEME_NAMES = ["no_weights", "weight_views", "weight_views_squared"]


def make_sample_weights(last_known_views, scheme_name):
    if scheme_name == "no_weights":
        return np.ones(len(last_known_views))
    if scheme_name == "weight_views":
        return last_known_views.to_numpy(dtype=float)
    if scheme_name == "weight_views_squared":
        return last_known_views.to_numpy(dtype=float) ** 2
    raise ValueError(f"unknown weight scheme: {scheme_name}")


def evaluate_lightgbm_growth_model(videos, splitter, feature_names, target_name, scheme_name):
    fold_results = []
    for fold_number, (fit_index, holdout_index) in enumerate(splitter.split(videos), start=1):
        fit_part = videos.iloc[fit_index]
        holdout_part = videos.iloc[holdout_index]

        training_growth = fit_part["growth_factor"].clip(lower=MINIMUM_GROWTH)
        if target_name == "log_growth":
            training_target = np.log(training_growth)
        else:
            training_target = training_growth

        model = lgb.LGBMRegressor(**LIGHTGBM_PARAMS)
        model.fit(
            fit_part[feature_names],
            training_target,
            sample_weight=make_sample_weights(fit_part["last_known_views"], scheme_name),
        )

        model_output = model.predict(holdout_part[feature_names])
        if target_name == "log_growth":
            predicted_growth = np.exp(model_output)
        else:
            predicted_growth = model_output

        share_below_minimum = float(np.mean(predicted_growth < MINIMUM_GROWTH))
        predicted_growth = np.maximum(predicted_growth, MINIMUM_GROWTH)
        predicted_views = holdout_part["last_known_views"].to_numpy() * predicted_growth
        actual_views = holdout_part["target_day30_views"]

        fold_results.append({
            "fold": fold_number,
            "rmse": compute_rmse(actual_views, predicted_views),
            "log_rmse": compute_rmse_on_log_scale(actual_views, predicted_views),
            "share_below_minimum": share_below_minimum,
        })
    return pd.DataFrame(fold_results)


all_experiment_tables = []
for target_name in TARGET_NAMES:
    for scheme_name in WEIGHT_SCHEME_NAMES:
        experiment_table = evaluate_lightgbm_growth_model(
            usable_train_videos, fold_splitter, ALL_FEATURES, target_name, scheme_name
        )
        experiment_table["experiment"] = f"{target_name} | {scheme_name}"
        all_experiment_tables.append(experiment_table)

experiment_results = pd.concat(all_experiment_tables, ignore_index=True)

raw_rmse_by_fold = experiment_results.pivot(index="fold", columns="experiment", values="rmse")
log_rmse_by_fold = experiment_results.pivot(index="fold", columns="experiment", values="log_rmse")

baseline_by_fold = baseline_results.set_index("fold")
raw_rmse_by_fold.insert(0, "baseline", baseline_by_fold["rmse_last_known_times_median"])
log_rmse_by_fold.insert(0, "baseline", baseline_by_fold["log_rmse_last_known_times_median"])

print(raw_rmse_by_fold.T.round(0))
print(raw_rmse_by_fold.agg(["mean", "std"]).T.round(0))
print(log_rmse_by_fold.agg(["mean", "std"]).T.round(3))
print(experiment_results.groupby("experiment")["share_below_minimum"].mean().round(4))

fold                                       1         2         3         4  \
experiment                                                                   
baseline                             87323.0   76439.0   75484.0   77673.0   
growth | no_weights                2711544.0  268171.0  165835.0  203837.0   
growth | weight_views               103472.0  104334.0   76222.0  184824.0   
growth | weight_views_squared        85022.0   51343.0   67323.0  148496.0   
log_growth | no_weights              82629.0   78011.0   66333.0   88126.0   
log_growth | weight_views            82519.0   79208.0   73747.0  112680.0   
log_growth | weight_views_squared    79662.0   51479.0   67982.0  165664.0   

fold                                       5  
experiment                                    
baseline                             55233.0  
growth | no_weights                1111148.0  
growth | weight_views                72183.0  
growth | weight_views_squared        60829.0  
log_growth | no

## 17. Feature Group Ablation

In [18]:
def evaluate_log_growth_model(videos, splitter, feature_names):
    fold_results = []
    for fold_number, (fit_index, holdout_index) in enumerate(splitter.split(videos), start=1):
        fit_part = videos.iloc[fit_index]
        holdout_part = videos.iloc[holdout_index]

        training_growth = fit_part["growth_factor"].clip(lower=MINIMUM_GROWTH)
        training_target = np.log(training_growth)

        model = lgb.LGBMRegressor(**LIGHTGBM_PARAMS)
        model.fit(fit_part[feature_names], training_target)

        predicted_growth = np.exp(model.predict(holdout_part[feature_names]))
        predicted_growth = np.maximum(predicted_growth, MINIMUM_GROWTH)
        predicted_views = holdout_part["last_known_views"].to_numpy() * predicted_growth
        actual_views = holdout_part["target_day30_views"]

        fold_results.append({
            "fold": fold_number,
            "rmse": compute_rmse(actual_views, predicted_views),
            "log_rmse": compute_rmse_on_log_scale(actual_views, predicted_views),
        })
    return pd.DataFrame(fold_results)


all_features_results = evaluate_log_growth_model(usable_train_videos, fold_splitter, ALL_FEATURES)

ablation_tables = [all_features_results.assign(experiment="all_features")]
for group_name in FEATURE_GROUPS:
    if group_name == "base":
        continue
    reduced_features = [name for name in ALL_FEATURES if name not in FEATURE_GROUPS[group_name]]
    group_results = evaluate_log_growth_model(usable_train_videos, fold_splitter, reduced_features)
    ablation_tables.append(group_results.assign(experiment=f"without_{group_name}"))

ablation_results = pd.concat(ablation_tables, ignore_index=True)

raw_rmse_by_fold = ablation_results.pivot(index="fold", columns="experiment", values="rmse")
log_rmse_by_fold = ablation_results.pivot(index="fold", columns="experiment", values="log_rmse")

print(log_rmse_by_fold.agg(["mean", "std"]).T.round(3).sort_values("mean"))
print(raw_rmse_by_fold.agg(["mean", "std"]).T.round(0).sort_values("mean"))

                           mean    std
experiment                            
all_features              0.283  0.037
without_engagement_rates  0.283  0.037
without_emotions          0.284  0.036
without_daily_counts      0.284  0.037
without_video             0.284  0.038
without_creator           0.289  0.039
without_momentum          0.299  0.035
                             mean      std
experiment                                
without_video             66383.0  18322.0
without_emotions          71866.0  17407.0
without_engagement_rates  73804.0  15959.0
all_features              74025.0  13312.0
without_momentum          76504.0  40410.0
without_daily_counts      82652.0  18881.0
without_creator           83032.0  34563.0


## 18. Final Model and Predictions

In [19]:
final_model = lgb.LGBMRegressor(**LIGHTGBM_PARAMS)

final_training_growth = usable_train_videos["growth_factor"].clip(lower=MINIMUM_GROWTH)
final_training_target = np.log(final_training_growth)

final_model.fit(usable_train_videos[ALL_FEATURES], final_training_target)

test_predicted_growth = np.exp(final_model.predict(test_videos_with_engagement[ALL_FEATURES]))
test_predicted_growth = np.maximum(test_predicted_growth, MINIMUM_GROWTH)

test_predictions = test_videos_with_engagement["last_known_views"] * test_predicted_growth
test_predictions = test_predictions.round().astype(int)
test_predictions = test_predictions.rename("target_day30_views")

feature_importance = pd.Series(
    final_model.feature_importances_, index=ALL_FEATURES
).sort_values(ascending=False)

print(test_predictions.describe())
print(feature_importance.head(15))

count    3.001000e+03
mean     2.564671e+04
std      1.971871e+05
min      8.000000e+00
25%      3.460000e+02
50%      6.890000e+02
75%      2.545000e+03
max      6.764122e+06
Name: target_day30_views, dtype: float64
last_day_views_share            313
last_two_days_views_share       246
creator_follower_count_gain     180
views_per_follower_day5         169
post_hour                       150
creator_video_count_gain        143
views_growth_day1_to_day5       143
likes_per_view_day5             138
creator_total_favorited_gain    100
collects_per_view_day5           97
disgust                          96
play_count_day1                  89
topic                            84
creator_following_count_day5     78
creator_video_count_day5         77
dtype: int32


## 19. Submission File

In [20]:
submission = pd.DataFrame({
    "video_id": test_videos_with_engagement["video_id"],
    "target_day30_views": test_predictions,
})

assert submission.shape[0] == sample_submission.shape[0]
assert list(submission.columns) == list(sample_submission.columns)
assert submission["video_id"].isin(sample_submission["video_id"]).all()
assert sample_submission["video_id"].isin(submission["video_id"]).all()
assert submission["video_id"].duplicated().sum() == 0
assert submission["target_day30_views"].isna().sum() == 0
assert (submission["target_day30_views"] >= 0).all()

submission = submission.set_index("video_id").loc[sample_submission["video_id"]].reset_index()

print(submission.shape)
print(submission.head())

submission.to_csv("baseline_submission.csv", index=False)

(3001, 2)
              video_id  target_day30_views
0  7400582591771938090                 246
1  7403167206978374958                 382
2  7435124823820356906                2086
3  7409800970143714606                 284
4  7410935177553333550                 260
